# llm-confidence — Usage Examples

This notebook demonstrates every feature of `llm-confidence`:

1. **Scalar field** — single category classification
2. **Three confidence metrics** — joint, mean, mean_nonzero
3. **Top alternatives** — what the model considered
4. **Token inspection** — see exactly which tokens were used
5. **Array field** — batch classification in one call
6. **Pydantic model auto-detection** — no need to specify `field=`
7. **Without specifying a field** — returns all fields
8. **Lower-level API** — parser, token helpers, normalizer

In [ ]:
import os
import sys
from pathlib import Path

os.environ["OPENAI_API_KEY"] = "sk-..."
os.environ["OPENAI_API_BASE"] = "https://api.openai.com/v1"

# When running from within the repo, ensure the repo root is on sys.path.
# In a pip-installed setup this cell is unnecessary.
_repo_root = str(Path("..").resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from enum import Enum

import litellm
from pydantic import BaseModel

from llm_confidence import extract_field_logprobs, FieldLogprob

## Setup

In [ ]:
MODEL = "gpt-4.1-mini"

SYSTEM_PROMPT = (
    "You are an assistant that classifies text into topics. "
    "Given a short description, classify it into exactly one category. "
    "Return JSON with a single key 'category'."
)


class CategoryEnum(str, Enum):
    health_and_wellness = "health and wellness"
    sports = "sports"
    technology = "technology"
    entertainment = "entertainment"
    science = "science"


class SingleCategory(BaseModel):
    category: CategoryEnum


class MultiCategory(BaseModel):
    categories: list[CategoryEnum]


CATEGORY_NAMES = [e.value for e in CategoryEnum]

SCALAR_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "topic_classification",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {"category": {"type": "string", "enum": CATEGORY_NAMES}},
            "required": ["category"],
            "additionalProperties": False,
        },
    },
}

ARRAY_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "multi_topic_classification",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "categories": {
                    "type": "array",
                    "items": {"type": "string", "enum": CATEGORY_NAMES},
                }
            },
            "required": ["categories"],
            "additionalProperties": False,
        },
    },
}


def call_llm(text: str, *, model=MODEL, response_format=None, system=None):
    return litellm.completion(
        model=model,
        messages=[
            {"role": "system", "content": system or SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        response_format=response_format or SCALAR_FORMAT,
        temperature=0,
        logprobs=True,
        top_logprobs=5,
        num_retries=2,
    )

---
## 1. Scalar field — basic usage

In [ ]:
resp = call_llm("Morning yoga and meditation session")
result = extract_field_logprobs(resp, field="category")

for value, fl in result.items():
    print(f"Category: {value}")
    print(f"  Joint probability:         {fl.joint_probability:.2%}")
    print(f"  Mean probability:          {fl.mean_probability:.2%}")
    print(f"  Mean non-zero probability: {fl.mean_nonzero_probability:.2%}")

---
## 2. Three metrics explained

| Metric | Formula | Best for |
|---|---|---|
| `joint_probability` | exp(sum of all logprobs) | Strictest — penalizes long values |
| `mean_probability` | exp(mean of all logprobs) | Comparing values with different token counts |
| `mean_nonzero_probability` | exp(mean of logprobs where logprob != 0) | **ENUM classification** — ignores deterministic tokens |

In [ ]:
# Run two texts: one obvious, one ambiguous
texts = [
    "Yoga and meditation retreat in the mountains",  # obvious: health and wellness
    "AI startup funding round announcement",         # ambiguous: technology? science?
]

for txn in texts:
    resp = call_llm(txn)
    result = extract_field_logprobs(resp, field="category")
    value, fl = next(iter(result.items()))

    print(f"\n{txn}")
    print(f"  → {value}")
    print(f"  joint:        {fl.joint_probability:8.2%}  (logprob: {fl.joint_logprob:.4f})")
    print(f"  mean:         {fl.mean_probability:8.2%}  (logprob: {fl.mean_logprob:.4f})")
    print(f"  mean_nonzero: {fl.mean_nonzero_probability:8.2%}  (logprob: {fl.mean_nonzero_logprob:.4f})")
    print(f"  tokens: {len(fl.tokens)} total, {sum(1 for t in fl.tokens if t.logprob != 0.0)} non-zero")

---
## 3. Token inspection

See exactly which tokens were included in the computation (and which were excluded).

In [ ]:
resp = call_llm("Morning yoga and meditation session")
result = extract_field_logprobs(resp, field="category")
fl = next(iter(result.values()))

print(f"Value: {fl.value!r}")
print(f"Tokens included ({len(fl.tokens)}):")
for t in fl.tokens:
    marker = "  " if t.logprob == 0.0 else "→ "
    print(f"  {marker}{t.token!r:25s}  logprob={t.logprob:10.6f}  prob={t.probability:.2%}")

print(f"\nRaw response content: {resp.choices[0].message.content!r}")
print(f"All logprobs tokens (including structural ones the lib EXCLUDED):")
for t in resp.choices[0].logprobs.content:
    print(f"  {t.token!r:25s}  logprob={t.logprob:10.6f}")

---
## 4. Top alternatives

See what the model considered at the most uncertain token position.

In [ ]:
resp = call_llm("AI startup funding round and product launch")
result = extract_field_logprobs(resp, field="category")
fl = next(iter(result.values()))

print(f"Chosen: {fl.value!r} ({fl.mean_nonzero_probability:.2%})")
print(f"\nTop alternatives at first uncertain token:")
for alt in fl.top_logprobs:
    print(f"  {alt.token!r:25s}  prob={alt.probability:.2%}  logprob={alt.logprob:.4f}")

---
## 5. Array field — batch classification

In [ ]:
batch_prompt = (
    "Classify each item:\n"
    "1. Morning yoga and meditation\n"
    "2. Yoga and meditation retreat\n"
    "3. New smartphone release\n"
    "4. Netflix documentary\n"
    "Return JSON with key 'categories' as an array, one category per item, in order."
)

resp = call_llm(
    batch_prompt,
    response_format=ARRAY_FORMAT,
    system=SYSTEM_PROMPT + " Return a JSON array of categories, one per item.",
)

print(f"Raw content: {resp.choices[0].message.content}\n")

result = extract_field_logprobs(resp, field="categories")
print(f"Results ({len(result)} categories):")
for value, fl in result.items():
    nz = f"{fl.mean_nonzero_probability:.2%}" if fl.mean_nonzero_probability else "N/A"
    print(f"  {value:30s}  joint={fl.joint_probability:.2%}  mean_nz={nz}")

---
## 6. Pydantic model auto-detection

Instead of `field="category"`, pass the Pydantic model and the lib finds enum fields automatically.

In [ ]:
# Scalar — auto-detects 'category' from SingleCategory.category: CategoryEnum
resp = call_llm("Fitness app and workout tracking")
result = extract_field_logprobs(resp, model=SingleCategory)

value, fl = next(iter(result.items()))
print(f"Auto-detected field 'category' from SingleCategory model")
print(f"  → {value} ({fl.joint_probability:.2%})")

In [ ]:
# Array — auto-detects 'categories' from MultiCategory.categories: list[CategoryEnum]
resp = call_llm(
    "Classify: 1. Ride-sharing app 2. Pharmacy and vitamins\nReturn categories array.",
    response_format=ARRAY_FORMAT,
    system=SYSTEM_PROMPT + " Return a JSON array of categories.",
)
result = extract_field_logprobs(resp, model=MultiCategory)

print(f"Auto-detected field 'categories' from MultiCategory model")
for value, fl in result.items():
    print(f"  → {value} ({fl.joint_probability:.2%})")

---
## 7. Without specifying a field — returns all fields

In [ ]:
resp = call_llm("Yoga and meditation retreat")
result = extract_field_logprobs(resp)  # no field= or model=

print("All fields returned:")
for value, fl in result.items():
    print(f"  {value!r:30s}  joint={fl.joint_probability:.2%}  tokens={[t.token for t in fl.tokens]}")

---
## 8. Lower-level API

For custom workflows (parse JSON without an LLM response, build your own metrics), use the internal modules directly.

In [ ]:
from llm_confidence._parser import (
    parse_json_spans,
    build_token_char_ranges,
    get_overlapping_indices,
    tokens_for_span,
)
from llm_confidence._converter import normalize_response

# 1. Parse JSON and get value + char range for every atomic value
parsed = parse_json_spans('{"category": "sports", "count": 2}')
print("parse_json_spans — each value has .value, .char_start, .char_end:")
print(f"  category: {parsed['category'].value!r}  chars [{parsed['category'].char_start}, {parsed['category'].char_end})")
print(f"  count:   {parsed['count'].value!r}  chars [{parsed['count'].char_start}, {parsed['count'].char_end})")

# 2. For arrays, each element is a _ValueSpan
arr_parsed = parse_json_spans('{"items": ["a", "b", "c"]}')
print(f"\n  items: {[s.value for s in arr_parsed['items']]}")

# 3. Normalize a response (works with any provider)
resp = call_llm("Yoga and meditation retreat")
norm = normalize_response(resp)
print(f"\nnormalize_response: content length={len(norm.content)}, tokens={len(norm.tokens)}")

# 4. Token helpers — use spans from the actual response content
parsed_resp = parse_json_spans(norm.content)
span = parsed_resp["category"]
ranges = build_token_char_ranges(norm.tokens)
token_infos = tokens_for_span(span.char_start, span.char_end, norm.tokens, ranges)
print(f"\ntokens_for_span('category'): {[t.token for t in token_infos]}")

---
## Summary

```python
from llm_confidence import extract_field_logprobs

# All these work:
result = extract_field_logprobs(resp, field="category")          # explicit field
result = extract_field_logprobs(resp, model=SingleCategory)      # auto-detect enum
result = extract_field_logprobs(resp, field="categories")        # array field
result = extract_field_logprobs(resp, model=MultiCategory)       # auto-detect list[Enum]
result = extract_field_logprobs(resp)                            # all fields

# Access metrics:
fl = result["health and wellness"]
fl.joint_probability           # strictest
fl.mean_probability            # fair comparison across token counts
fl.mean_nonzero_probability    # best for ENUM classification
fl.tokens                      # inspect individual tokens
fl.top_logprobs                # alternatives the model considered
```